In [ ]:
import os
import sys

sys.path.append("/home/justin/code/point-to-pose/")
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import glob
import copy
import open3d as o3d
from omegaconf import OmegaConf


import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Increase the embedding limit (set to 50 MB)
plt.rcParams["animation.embed_limit"] = 500

import torch
import torch.nn.functional as F

import point2pose.pipeline.pipeline as pipeline_module
from point2pose.data_types.frame import Frame

In [ ]:
config_pth = "/home/justin/code/point-to-pose/configs/pipeline/pipeline_test.yaml"
data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/edamame_box"

In [ ]:


# Function to load RGB images
def load_rgb_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load RGB images from the /rgb subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.jpg', '.jpeg', '.png'])
    
    Returns:
        List[np.ndarray]: List of RGB images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png']
    
    rgb_folder = os.path.join(folder_path, 'rgb')
    if not os.path.exists(rgb_folder):
        raise FileNotFoundError(f"RGB folder not found: {rgb_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(rgb_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            img = cv2.imread(file_path)
            if img is not None:
                # Convert BGR to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img_rgb)
    
    return images

# Function to load depth images
def load_depth_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load depth images from the /depth subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /depth subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.tiff', '.tif'])
    
    Returns:
        List[np.ndarray]: List of depth images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.png', '.tiff', '.tif']
    
    depth_folder = os.path.join(folder_path, 'depth')
    if not os.path.exists(depth_folder):
        raise FileNotFoundError(f"Depth folder not found: {depth_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(depth_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load depth image (usually 16-bit)
            img = cv2.imread(file_path, cv2.IMREAD_ANYDEPTH)
            if img is not None:
                images.append(img)
    
    return images

# Function to load mask images
def load_mask_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load mask images from the /masks subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /masks subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.jpg', '.jpeg'])
    
    Returns:
        List[np.ndarray]: List of mask images as numpy arrays (binary or grayscale)
    """
    if file_extensions is None:
        file_extensions = ['.png', '.jpg', '.jpeg']
    
    masks_folder = os.path.join(folder_path, 'masks')
    if not os.path.exists(masks_folder):
        raise FileNotFoundError(f"Masks folder not found: {masks_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(masks_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load mask as grayscale
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    
    return images

# Function to load all image types together
def load_all_images(folder_path: str) -> Dict[str, List[np.ndarray]]:
    """
    Load RGB, depth, and mask images from the specified folder structure.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[np.ndarray]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of images
    """
    result = {}
    
    try:
        result['rgb'] = load_rgb_images(folder_path)
        print(f"Loaded {len(result['rgb'])} RGB images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['rgb'] = []
    
    try:
        result['depth'] = load_depth_images(folder_path)
        print(f"Loaded {len(result['depth'])} depth images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['depth'] = []
    
    try:
        result['masks'] = load_mask_images(folder_path)
        print(f"Loaded {len(result['masks'])} mask images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['masks'] = []
    
    return result

# Function to get file paths without loading images
def get_image_paths(folder_path: str) -> Dict[str, List[str]]:
    """
    Get file paths for RGB, depth, and mask images without loading them.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[str]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of file paths
    """
    result = {}
    
    # RGB paths
    rgb_folder = os.path.join(folder_path, 'rgb')
    if os.path.exists(rgb_folder):
        rgb_files = []
        for ext in ['.jpg', '.jpeg', '.png']:
            pattern = os.path.join(rgb_folder, f"*{ext}")
            rgb_files.extend(glob.glob(pattern))
        result['rgb'] = sorted(rgb_files)
    else:
        result['rgb'] = []
    
    # Depth paths
    depth_folder = os.path.join(folder_path, 'depth')
    if os.path.exists(depth_folder):
        depth_files = []
        for ext in ['.png', '.tiff', '.tif']:
            pattern = os.path.join(depth_folder, f"*{ext}")
            depth_files.extend(glob.glob(pattern))
        result['depth'] = sorted(depth_files)
    else:
        result['depth'] = []
    
    # Mask paths
    masks_folder = os.path.join(folder_path, 'masks')
    if os.path.exists(masks_folder):
        mask_files = []
        for ext in ['.png', '.jpg', '.jpeg']:
            pattern = os.path.join(masks_folder, f"*{ext}")
            mask_files.extend(glob.glob(pattern))
        result['masks'] = sorted(mask_files)
    else:
        result['masks'] = []
    
    return result

def wait_for_click_and_get_pixel(image):
    """
    Display an image in a Jupyter notebook and wait for a mouse click.
    Returns (x, y) pixel coordinates of the click.
    """
    coords = []

    def onclick(event):
        # Only respond to left mouse button clicks
        if event.button == 1 and event.xdata is not None and event.ydata is not None:
            coords.append((int(event.xdata), int(event.ydata)))
            plt.close()  # Close the figure after click

    fig, ax = plt.subplots()
    ax.imshow(image)
    ax.set_title("Click on the image")
    cid = fig.canvas.mpl_connect('button_press_event', onclick)

    plt.show()

    if coords:
        return coords[0]
    else:
        return None

def load_camera_param(folder_path: str) -> Dict[str, np.ndarray]:
    """
    Load camera parameters from the specified folder.
    
    Args:
        folder_path (str): Path to the main folder containing /camera_param subdirectory
    
    Returns:
        Dict[str, np.ndarray]: Dictionary with keys 'K', 'D' containing camera parameters
    """
    # load txt
    cam_param_path = os.path.join(folder_path, 'cam_K.txt')
    # if not os.path.exists(camera_param_folder):
    #     raise FileNotFoundError(f"Camera parameter folder not found: {camera_param_folder}")
    
    # load the camera parameters from the camera_param subdirectory
    # with open(cam_param_path, 'r') as file:
    #     lines = [line.strip() for line in file]
    return np.loadtxt(cam_param_path)


In [ ]:
### plotting related functions ###

def draw_points_on_image(image, points, colors):
    # if points are tensor
    # print(points.shape)
    if isinstance(points, torch.Tensor):
        points = points.cpu().numpy()
        
    for i in range(points.shape[0]):
        cv2.circle(
            image,
            points[i, :].astype(int).reshape(2),
            radius=5,
            color=colors[i],
            thickness=-1,
        )

def get_n_colors(n):
        cmap = plt.get_cmap("RdYlGn")  # or 'tab20', 'jet', etc.
        colors = [tuple(int(c * 255) for c in cmap(i / n)[:3]) for i in range(n)]
        return colors

def get_n_uncertainty_colors(uncertainties, u_min=0.0, u_max=1.0, inverse=False):
        cmap = plt.get_cmap("jet")  # or 'tab20', 'jet', etc.
        norm_uncertainties = (uncertainties - u_min) / (u_max - u_min + 1e-8)
        if inverse:
            norm_uncertainties = 1 - norm_uncertainties
        colors = [tuple(int(c * 255) for c in cmap(u)[:3]) for u in norm_uncertainties]
        return colors
        
def visualize_out_frames_slider(out_frames):
    # Create slider
    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(out_frames)-1,
        step=1,
        description='Frame:',
        continuous_update=False
    )

    # Create output widget
    output = widgets.Output()

    def on_value_change(change):
        with output:
            output.clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            plt.imshow(out_frames[change['new']])
            plt.title(f"Frame {change['new']}")
            plt.axis('off')
            plt.show()

    slider.observe(on_value_change, names='value')

    # Display widgets
    display(slider, output)
    # Trigger initial display
    on_value_change({'new': 0})


def visualize_out_frames_animation(out_frames):

    # Your existing animation code
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title("Image Sequence")

    def animate(frame):
        ax.clear()
        ax.imshow(out_frames[frame])
        ax.set_title(f"Frame {frame}")
        ax.axis('off')
        return ax,

    anim = animation.FuncAnimation(
        fig, animate, frames=len(out_frames), 
        interval=100, repeat=True, blit=False
    )

    display(HTML(anim.to_jshtml()))
    plt.close(fig)

    return anim


In [ ]:

def _apply_tf(points: np.ndarray, tf: np.ndarray) -> np.ndarray:
    """Apply a 4x4 homogeneous transform to Nx3 points."""
    points = np.asarray(points)
    assert points.ndim == 2 and points.shape[1] == 3, "points must be (N,3)"
    assert tf.shape == (4, 4), "tf must be a 4x4 homogeneous matrix"
    homog = np.c_[points, np.ones(len(points))]
    out = homog @ tf.T
    return out[:, :3]


def _auto_palette(n: int):
    # deterministic-ish palette in [0..255]
    if n <= 0:
        return np.zeros((0, 3), dtype=np.uint8)
    # golden ratio trick in HSV
    h = (np.arange(n) * 0.61803398875) % 1.0
    s = np.full(n, 0.65)
    v = np.full(n, 0.95)
    # hsv -> rgb
    i = np.floor(h * 6).astype(int)
    f = h * 6 - i
    p = v * (1 - s)
    q = v * (1 - f * s)
    t = v * (1 - (1 - f) * s)
    rgb = np.zeros((n, 3))
    idx = (i % 6 == 0); rgb[idx] = np.stack([v[idx], t[idx], p[idx]], 1)
    idx = (i % 6 == 1); rgb[idx] = np.stack([q[idx], v[idx], p[idx]], 1)
    idx = (i % 6 == 2); rgb[idx] = np.stack([p[idx], v[idx], t[idx]], 1)
    idx = (i % 6 == 3); rgb[idx] = np.stack([p[idx], q[idx], v[idx]], 1)
    idx = (i % 6 == 4); rgb[idx] = np.stack([t[idx], p[idx], v[idx]], 1)
    idx = (i % 6 == 5); rgb[idx] = np.stack([v[idx], p[idx], q[idx]], 1)
    return (rgb * 255).astype(np.uint8)

def _normalize_colors(c, n):
    """
    Accept None, CSS string, single RGB (3,), or Nx3 in [0..1] or [0..255].
    Return:
      - for plotly: list of "rgb(r,g,b)" strings or a single CSS string
      - for k3d: packed uint32 per-point
    """
    if c is None:
        return None
    c = np.asarray(c)
    if c.ndim == 1 and c.size == 3:
        c = np.tile(c[None, :], (n, 1))
    if c.ndim == 2 and c.shape[1] == 3:
        if c.max() <= 1.0:
            c = (c * 255).astype(np.uint8)
        else:
            c = c.astype(np.uint8)
        return c
    # let plotly handle strings/lists of strings; k3d path handles ints
    return c

def _plotly_colorize(c_uint8_or_str, n, fallback):
    if c_uint8_or_str is None:
        return fallback
    if isinstance(c_uint8_or_str, np.ndarray) and c_uint8_or_str.ndim == 2:
        return [f"rgb({r},{g},{b})" for r, g, b in c_uint8_or_str]
    return c_uint8_or_str  # string or list of strings

def _k3d_pack_rgb(c_uint8, n, fallback_hex):
    if c_uint8 is None:
        return np.full(n, fallback_hex, dtype=np.uint32)
    if isinstance(c_uint8, np.ndarray) and c_uint8.ndim == 2:
        c = c_uint8.astype(np.uint32)
        return (c[:,0] << 16) | (c[:,1] << 8) | c[:,2]
    # already an int or array of ints
    return np.asarray(c_uint8, dtype=np.uint32)

# ---------- 1) Single cloud ----------
def vis_point_cloud(
    pts: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    color=None,                 # None / str / (3,) / Nx3 in [0..1] or [0..255]
    title: str = "Point Cloud",
):
    assert pts.ndim == 2 and pts.shape[1] == 3
    n = pts.shape[0]
    c = _normalize_colors(color, n)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go
        col = _plotly_colorize(c, n, "royalblue")
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col),
            name="cloud"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display
        cols = _k3d_pack_rgb(c, n, 0x4169E1)  # royalblue
        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(pts.astype(np.float32), colors=cols, point_size=point_size*0.01, shader='3d')
        plot.camera_auto_fit = True
        return display(plot)

    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

def vis_reg_points(
    src: np.ndarray,
    trg: np.ndarray,
    tf: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    src_color=None,            # None, single color string (e.g. 'red'), or per-point Nx3 in [0,1] or 0..255
    trg_color=None,
    title: str = "Registered point clouds (src→trg)"
):
    """
    Visualize two point clouds after applying tf to src.
    Args:
        src, trg: (N,3) and (M,3) float arrays.
        tf: (4,4) homogeneous transform that maps src -> trg frame.
        backend: 'plotly' (inline, easy) or 'k3d' (very fast for huge clouds).
        point_size: marker size (Plotly) or glyph size (k3d).
        src_color, trg_color: None, single color string/int, or per-point Nx3.
    """
    src_t = _apply_tf(src, tf)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go

        def _to_plotly_color(arr, fallback):
            if arr is None:
                return fallback
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:  # single RGB triplet
                arr = np.tile(arr, (1,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                # normalize if in 0..1
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                else:
                    arr = arr.astype(np.uint8)
                return [f"rgb({r},{g},{b})" for r, g, b in arr]
            # otherwise assume a CSS color string or list Plotly can handle
            return arr

        src_col = _to_plotly_color(src_color, "red")
        trg_col = _to_plotly_color(trg_color, "royalblue")

        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=trg[:,0], y=trg[:,1], z=trg[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=trg_col),
            name="target (trg)"
        ))
        fig.add_trace(go.Scatter3d(
            x=src_t[:,0], y=src_t[:,1], z=src_t[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=src_col),
            name="source transformed (src·tf)"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
            legend=dict(itemsizing="constant")
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display

        def _to_k3d_colors(arr, n, fallback_hex):
            if arr is None:
                return np.full(n, fallback_hex, dtype=np.uint32)
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:
                arr = np.tile(arr, (n,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                arr = arr.astype(np.uint8)
                return ((arr[:,0].astype(np.uint32) << 16) |
                        (arr[:,1].astype(np.uint32) << 8) |
                         arr[:,2].astype(np.uint32))
            # single packed int color or array of packed ints
            return arr.astype(np.uint32)

        trg_pts = trg.astype(np.float32)
        src_pts = src_t.astype(np.float32)
        trg_cols = _to_k3d_colors(trg_color, len(trg_pts), 0x4169E1)  # royalblue
        src_cols = _to_k3d_colors(src_color, len(src_pts), 0xFF0000)  # red

        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(trg_pts, colors=trg_cols, point_size=point_size*0.01, shader='3d', name='trg')
        plot += k3d.points(src_pts, colors=src_cols, point_size=point_size*0.01, shader='3d', name='src·tf')
        plot.camera_auto_fit = True
        display(plot)
    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

In [ ]:
### load data ###
data = load_all_images(data_path)
data['K'] = load_camera_param(data_path)

In [ ]:
# initialize the first image 
rgb_image_first = data["rgb"][0]
depth_image_first = data["depth"][0]
mask = data["masks"][0]


plt.imshow(mask)
plt.show()

In [ ]:
# import importlib
# # from point2pose.pipeline.pipeline import Pipeline
# # import point2pose.pipeline.pipeline as pipeline_module

# importlib.reload(pipeline_module)

# ### Initialize tapir tracker ###

# cfg = OmegaConf.load(config_pth)

# pipeline = pipeline_module.Pipeline(cfg)

# pipeline.add_user_points([[200,200]], [1])

In [ ]:
from point2pose.utils.camera import convert_pixel_within_mask_to_world

pcd, valid = convert_pixel_within_mask_to_world(mask=data['masks'][0], 
cam_intrinsics=data['K'], depth_image=data['depth'][0], depth_factor=1000.0)

vis_point_cloud(pcd)


In [ ]:
import importlib
# from point2pose.pipeline.pipeline import Pipeline
# import point2pose.pipeline.pipeline as pipeline_module
# from IPython import autoreload
# autoreload.autoreload = 2

# %load_ext autoreload
# %autoreload 2

importlib.reload(pipeline_module)

### Initialize tapir tracker ###

cfg = OmegaConf.load(config_pth)

pipeline = pipeline_module.Pipeline(cfg)

pipeline.add_user_points([[200,200]], [1])

# loop through the rest of the images
out_frames = []


# for i in range(len(data["rgb"]))
for i in range(5):  # just do one frame for testing
    frame = Frame(
        rgb=data["rgb"][i],
        depth=data["depth"][i],
        mask=data["masks"][i],
        intrinsics=data["K"],
        depth_factor=1000.0,
        id=i,)


    pose = pipeline.step(frame)

     # Visualize the current frame with tracking information
    # rgb_image_vis = rgb_image.copy()
    # if i ==1:
        # vis_point_cloud(pipeline.prev3d_way_before)
        # vis_point_cloud(pipeline.prev3d_before)
        # vis_point_cloud(pipeline.prev3d_after)
        # vis_point_cloud(pipeline.curr3d_before)
        # vis_point_cloud(pipeline.curr3d_after)
    # draw_points_on_image(rgb_image_vis, tracks, get_n_colors(len(tracks)))
    # out_frames.append(rgb_image_vis)
    
    # vis_point_cloud(tracks_3d[valid_idx])
    # vis_reg_points(src=all_3d[all_valid], trg=tracks_3d[valid_idx], tf=np.eye(4), src_color=color[all_valid], trg_color=np.array([0,255,0]), point_size=5.0)

In [ ]:
visualize_out_frames_slider(out_frames)

In [ ]:
anim = visualize_out_frames_animation(out_frames)